<a href="https://colab.research.google.com/github/PitchieRichie/EcoMobility5442/blob/main/EcoMove_analysis%205443.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COM7021: EcoMove Mobility

 Each section builds one figure, followed by the statistics that support its findings. Section C re-draws Figures 1 to 8 as **interactive charts: hover over any bar, point or cell to read the story behind it**.

**How to run in Google Colab**
1. *Runtime > Run all* (or run the cells from top to bottom).
2. When asked, **upload** `_5442__EcoMoveMobilityDV.xlsx` (a *Choose files* button appears under the data cell).
3. Static figures appear under each section A.x; interactive figures under C.x; the dashboard is at the bottom.



## 0. Setup: install and load the data file

In [ ]:
# Colab already has pandas, matplotlib, seaborn, scipy and plotly. This adds Dash for the dashboard (about 30 seconds).
!pip -q install "dash>=2.11" statsmodels openpyxl

In [ ]:
import os
DATA = "_5442__EcoMoveMobilityDV.xlsx"
if not os.path.exists(DATA):
    try:
        from google.colab import files          # only exists inside Google Colab
        print("Choose _5442__EcoMoveMobilityDV.xlsx to upload ...")
        DATA = next(iter(files.upload()))       # name of the uploaded file
    except ImportError:
        raise FileNotFoundError("Put _5442__EcoMoveMobilityDV.xlsx in the same folder as this notebook.")
print("Using data file:", DATA)

In [ ]:
# Quick health check: library versions and that the Excel file has the expected columns (prints a clear message if not)
import pandas as pd, numpy as np, plotly
print("pandas", pd.__version__, "| numpy", np.__version__, "| plotly", plotly.__version__)
_chk = pd.read_excel(DATA)
_need = ["Record_ID", "Date", "City", "Country", "Transport_Mode", "Route_Type", "Weather", "Event_Day", "Passengers", "Trips", "Average_Delay_Minutes",
         "On_Time_Percentage", "Ticket_Revenue_EUR", "Operating_Cost_EUR", "CO2_Saved_KG", "Energy_Consumption_KWh", "Customer_Rating", "Accessibility_Complaints"]
_missing = [c for c in _need if c not in _chk.columns]
assert not _missing, f"The Excel file is missing columns {_missing}. Is it the right file (_5442__EcoMoveMobilityDV.xlsx)?"
print(f"OK: {len(_chk):,} rows, all {len(_need)} expected columns found. Run the cells below in order, top to bottom.")

# A. Static figures and statistics (Figures A, B and 1 to 8)

## A.1 Setup, loading, validation and cleaning (Figures A and B)

In [ ]:
"""COM7021 - EcoMove Mobility: data preparation and static visualisations.
Run:  python eco_analysis.py
Outputs PNG figures to ./figs and a cleaned CSV to ./eco_clean.csv"""
import sys, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from matplotlib.ticker import FuncFormatter

# DATA (the Excel file name) is set in the setup cell above
FIG = "figs/"
IN_NOTEBOOK = "ipykernel" in sys.modules   # True in Colab/Jupyter: charts are displayed as well as saved
import os; os.makedirs(FIG, exist_ok=True)

# Okabe-Ito colour-blind-safe palette
OI = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({"axes.titleweight": "bold", "axes.titlesize": 12, "figure.dpi": 100,
                     "axes.spines.top": False, "axes.spines.right": False})

# ---------------------------------------------------------------- 1. LOAD & INSPECT
df = pd.read_excel(DATA)
print("Shape:", df.shape)
print(df.dtypes, "\n")
print("Missing values per column:\n", df.isna().sum().sum(), "total")
print("Duplicate rows:", df.duplicated().sum(), "| duplicate Record_IDs:", df.Record_ID.duplicated().sum())

# ---------------------------------------------------------------- 2. CLEAN & ENGINEER
# Dates are stored as Excel serial numbers -> convert to datetime
df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30")
df["Year"] = df.Date.dt.year
df["Month"] = df.Date.dt.to_period("M").dt.to_timestamp()
df["Event_Day"] = df["Event_Day"].map({"Yes": True, "No": False})
for c in ["City", "Country", "Transport_Mode", "Route_Type", "Weather"]:
    df[c] = df[c].astype("category")

# Derived KPIs
df["Net_Result_EUR"] = df.Ticket_Revenue_EUR - df.Operating_Cost_EUR
df["Revenue_per_Pax"] = df.Ticket_Revenue_EUR / df.Passengers
df["Cost_per_Pax"] = df.Operating_Cost_EUR / df.Passengers
df["CO2_per_Pax"] = df.CO2_Saved_KG / df.Passengers
df["Complaints_per_1000"] = df.Accessibility_Complaints / df.Passengers * 1000

# Validation checks
print("\nDate range:", df.Date.min().date(), "to", df.Date.max().date())
print("On-time % in [0,100]:", df.On_Time_Percentage.between(0, 100).all())
print("Rating in [1,5]:", df.Customer_Rating.between(1, 5).all())
raw_num = ["Passengers","Trips","Average_Delay_Minutes","On_Time_Percentage","Ticket_Revenue_EUR","Operating_Cost_EUR","CO2_Saved_KG","Energy_Consumption_KWh","Customer_Rating","Accessibility_Complaints"]
print("Negative values in source measures:", (df[raw_num] < 0).sum().sum())
print("Zero energy rows by mode:", df[df.Energy_Consumption_KWh == 0].Transport_Mode.value_counts().to_dict())
print("City->Country consistent:", (df.groupby("City", observed=True).Country.nunique() == 1).all())
Q1, Q3 = df.Passengers.quantile([.25, .75]); IQR = Q3 - Q1
print("Passenger outliers (1.5xIQR):", (df.Passengers > Q3 + 1.5 * IQR).sum(), "-> retained (genuine large-network days)")
df.to_csv("eco_clean.csv", index=False)

# helper for tidy aggregation
def agg(by):
    g = df.groupby(by, observed=True).agg(
        Passengers=("Passengers", "sum"), Trips=("Trips", "sum"),
        Delay=("Average_Delay_Minutes", "mean"), OnTime=("On_Time_Percentage", "mean"),
        Revenue=("Ticket_Revenue_EUR", "sum"), Cost=("Operating_Cost_EUR", "sum"),
        CO2=("CO2_Saved_KG", "sum"), Energy=("Energy_Consumption_KWh", "sum"),
        Rating=("Customer_Rating", "mean"), Complaints=("Accessibility_Complaints", "sum"))
    g["Net"] = g.Revenue - g.Cost
    g["Margin"] = g.Net / g.Revenue * 100
    g["Compl_per_1000"] = g.Complaints / g.Passengers * 1000
    g["CO2_per_Pax"] = g.CO2 / g.Passengers
    return g

def save(name):
    plt.tight_layout(); plt.savefig(FIG + name, dpi=200, bbox_inches="tight")
    plt.show() if IN_NOTEBOOK else plt.close()    # show inline in Colab/Jupyter; as a plain script just save to figs/

millions = FuncFormatter(lambda x, _: f"{x/1e6:.1f}M")

In [ ]:
# stats_tests.py: setup for the tests
"""COM7021 - supporting statistical tests and quantified scenarios. Run: python stats_tests.py"""
import pandas as pd, numpy as np
from scipy import stats
import statsmodels.formula.api as smf

d = pd.read_csv("eco_clean.csv")
D = "Average_Delay_Minutes"
df.head(4)

## A.2 Figure 1: passengers per record over time (line and bar panels)

In [ ]:
# FIG 1  TIME SERIES (normalised per record)
# The data are route-day records, and the number of records per month varies (188-228), so monthly TOTALS partly
# reflect record coverage. Demand is therefore shown as mean passengers per record.
m = df.groupby("Month").agg(Records=("Passengers", "size"), Pax=("Passengers", "mean"), Delay=("Average_Delay_Minutes", "mean"))
m["Roll3"] = m.Pax.rolling(3).mean()
fig, ax = plt.subplots(3, 1, figsize=(10, 7.6), sharex=True, gridspec_kw={"height_ratios": [2, 1, 1]})
ax[0].plot(m.index, m.Pax, color=OI[5], lw=1.5, marker="o", ms=4, label="Mean passengers per record")
ax[0].plot(m.index, m.Roll3, color=OI[0], lw=3, label="3-month rolling average")
ax[0].axhline(df.Passengers.mean(), color="k", ls="--", lw=1); ax[0].text(m.index[0], df.Passengers.mean() + 150, "overall mean", fontsize=8)
ax[0].set_ylabel("Passengers per record"); ax[0].set_ylim(5000, 11500); ax[0].legend(loc="upper right", frameon=False)
ax[0].set_title("Fig 1. Passengers per record show no clear year-on-year decline")
ax[1].bar(m.index, m.Records, width=20, color=OI[4]); ax[1].set_ylabel("Records / month"); ax[1].set_ylim(150, 240)
ax[2].bar(m.index, m.Delay, width=20, color=OI[3]); ax[2].set_ylabel("Avg delay (min)"); ax[2].set_ylim(3, 5.5)
ax[2].axhline(df.Average_Delay_Minutes.mean(), color="k", ls="--", lw=1)
save("fig1_timeseries.png")

In [ ]:
# stats_tests.py: is the 2025 fall in totals real?
# 9. Is the 2024->2025 fall in total passengers real? Compare record counts and passengers per record
d["Year"] = pd.to_datetime(d.Date).dt.year
y = d.groupby("Year").Passengers.agg(["size", "sum", "mean"])
print("\nRecords / total / mean per record by year:\n", y.round(0))
print(f"Total change {(y['sum'][2025]/y['sum'][2024]-1)*100:.1f}%, record-count change {(y['size'][2025]/y['size'][2024]-1)*100:.1f}%, "
      f"mean-per-record change {(y['mean'][2025]/y['mean'][2024]-1)*100:.1f}%")
a, b2 = d[d.Year == 2024].Passengers, d[d.Year == 2025].Passengers
print("Mann-Whitney 2024 vs 2025 passengers per record: p = %.2f" % stats.mannwhitneyu(a, b2).pvalue)
rng = np.random.default_rng(1)
boot = [rng.choice(b2.values, len(b2)).sum() / rng.choice(a.values, len(a)).sum() - 1 for _ in range(5000)]
print("Bootstrap 95%% CI for change in total passengers: [%.1f%%, %.1f%%]" % tuple(np.percentile(boot, [2.5, 97.5]) * 100))

## A.3 Figure 2: passengers by city (sorted bar chart)

In [ ]:
# FIG 2  COMPARISON (cities)
c = agg(["City", "Country"]).reset_index().sort_values("Passengers")
cols = dict(zip(sorted(c.Country.unique()), OI[:6]))
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.barh(c.City, c.Passengers, color=c.Country.map(cols))
for i, v in enumerate(c.Passengers): ax.text(v + 4e4, i, f"{v/1e6:.2f}M", va="center", fontsize=9)
ax.xaxis.set_major_formatter(millions); ax.set_xlabel("Total passengers 2024-25")
ax.set_title("Fig 2. Paris and Berlin each carry ~11% of all passengers")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=v) for v in cols.values()], labels=list(cols), title="Country",
          loc="lower right", frameon=False)
save("fig2_city_bar.png")

## A.4 Figure 3: correlation heatmap

In [ ]:
# FIG 3  CORRELATION HEATMAP
num = ["Passengers", "Trips", "Average_Delay_Minutes", "On_Time_Percentage", "Ticket_Revenue_EUR", "Operating_Cost_EUR",
       "CO2_Saved_KG", "Energy_Consumption_KWh", "Customer_Rating", "Accessibility_Complaints"]
lab = ["Passengers", "Trips", "Delay (min)", "On-time %", "Revenue", "Op. cost", "CO2 saved", "Energy", "Rating", "Access. compl."]
cm = df[num].corr(); cm.index = cm.columns = lab
fig, ax = plt.subplots(figsize=(8.5, 6.8))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True, linewidths=.5,
            cbar_kws={"shrink": .75, "label": "Pearson r"}, ax=ax, annot_kws={"size": 8})
ax.set_title("Fig 3. Two clusters: volume metrics move together; delay and satisfaction move oppositely")
save("fig3_corr.png")

In [ ]:
# stats_tests.py: regression of rating on delay
# 3. Regression: rating ~ delay
m = smf.ols("Customer_Rating ~ Average_Delay_Minutes", data=d).fit()
b = m.params.iloc[1]; ci = m.conf_int().iloc[1]
print(f"Rating ~ delay: slope={b:.3f} per min, 95% CI [{ci[0]:.3f}, {ci[1]:.3f}], R2={m.rsquared:.2f}, p={m.pvalues.iloc[1]:.2e}")

In [ ]:
# stats_tests.py: variance explained by each factor
# 10. How much variation does each factor account for? (R-squared from one-factor OLS; association only)
for outcome in ["Average_Delay_Minutes", "On_Time_Percentage", "Customer_Rating"]:
    r = {f: smf.ols(f"{outcome} ~ C({f})", data=d).fit().rsquared for f in ["Transport_Mode", "Weather", "Event_Day", "City"]}
    full = smf.ols(f"{outcome} ~ C(Transport_Mode)+C(Weather)+C(Event_Day)+C(City)", data=d).fit().rsquared
    print(outcome, {k: round(v, 3) for k, v in r.items()}, "combined", round(full, 3))

## A.5 Figure 4: delay by weather (box plot)

In [ ]:
# FIG 4  DISTRIBUTION (box plot)
order = df.groupby("Weather", observed=True).Average_Delay_Minutes.median().sort_values().index
wcol = {w: (OI[3] if w == "Snow" else OI[0]) for w in order}
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="Weather", y="Average_Delay_Minutes", order=order, palette=wcol, hue="Weather", legend=False,
            flierprops=dict(markersize=2, alpha=.4), ax=ax)
ax.set_xlabel(""); ax.set_ylabel("Average delay (minutes)")
for i, w in enumerate(order):
    ax.text(i, 13.2, f"n={int((df.Weather==w).sum())}", ha="center", fontsize=8)
ax.set_ylim(-0.5, 14)
ax.set_title("Fig 4. Snow more than doubles typical delays versus clear weather")
save("fig4_box_weather.png")

In [ ]:
# stats_tests.py: weather effect on delay
# 1. Weather effect on delay: Kruskal-Wallis (non-parametric) + Welch t-test snow vs clear
groups = [g[D].values for _, g in d.groupby("Weather")]
H, p = stats.kruskal(*groups); print(f"Kruskal-Wallis weather: H={H:.1f}, p={p:.2e}")
sn, cl = d[d.Weather == "Snow"][D], d[d.Weather == "Clear"][D]
t, p = stats.ttest_ind(sn, cl, equal_var=False)
diff = sn.mean() - cl.mean(); se = np.sqrt(sn.var() / len(sn) + cl.var() / len(cl))
print(f"Snow vs Clear: diff={diff:.2f} min, 95% CI [{diff-1.96*se:.2f}, {diff+1.96*se:.2f}], Welch t={t:.1f}, p={p:.2e}, n={len(sn)}/{len(cl)}")

In [ ]:
# stats_tests.py: event-day effect on delay
# 2. Event-day effect on delay
ev, ne = d[d.Event_Day == True][D], d[d.Event_Day == False][D]
t, p = stats.ttest_ind(ev, ne, equal_var=False)
diff = ev.mean() - ne.mean(); se = np.sqrt(ev.var() / len(ev) + ne.var() / len(ne))
print(f"Event vs non-event delay: diff={diff:.2f} min, 95% CI [{diff-1.96*se:.2f}, {diff+1.96*se:.2f}], t={t:.1f}, p={p:.2e}, n={len(ev)}/{len(ne)}")

In [ ]:
# stats_tests.py: snow records; Berlin event-day share
# 7. Snow: share of records / bus-tram-P&R snow delay premium
print("Snow records", len(sn), "share", round(len(sn) / len(d) * 100, 1))
# 8. Event days: berlin share
e = d[d.Event_Day == True]; print((e.groupby("City").Passengers.sum() / e.Passengers.sum()).sort_values(ascending=False).head(3).round(3).to_dict())

## A.6 Figure 5: mode performance scorecard

In [ ]:
# FIG 5  KPI SCORECARD (performance indicators)
mo = agg("Transport_Mode")
mo["Rev_per_Pax"] = mo.Revenue / mo.Passengers; mo["Cost_per_Pax"] = mo.Cost / mo.Passengers
sc = mo[["Passengers", "OnTime", "Delay", "Rating", "Margin", "Compl_per_1000", "CO2_per_Pax"]].copy()
sc = sc.sort_values("Passengers", ascending=False)
disp = pd.DataFrame({
    "Passengers\n(M)": sc.Passengers / 1e6, "On-time %": sc.OnTime, "Avg delay\n(min)": sc.Delay, "Rating\n(1-5)": sc.Rating,
    "Margin %": sc.Margin.clip(lower=-100), "Access. compl.\nper 1,000 pax": sc.Compl_per_1000, "CO2 saved\n(kg/pax)": sc.CO2_per_Pax})
good_high = [True, True, False, True, True, False, True]
score = disp.copy()
for col, gh in zip(disp.columns, good_high):
    z = (disp[col] - disp[col].mean()) / disp[col].std(); score[col] = z if gh else -z
fmt = disp.copy().astype(object)
fmt.iloc[:, 0] = disp.iloc[:, 0].map("{:.1f}".format)
for j, f in [(1, "{:.1f}"), (2, "{:.1f}"), (3, "{:.2f}"), (5, "{:.2f}"), (6, "{:.3f}")]: fmt.iloc[:, j] = disp.iloc[:, j].map(f.format)
fmt.iloc[:, 4] = [f"{v:,.0f}" for v in sc.Margin]
fig, ax = plt.subplots(figsize=(11.5, 4.6))
sns.heatmap(score, annot=fmt.values, fmt="", cmap="BrBG", center=0, vmin=-2, vmax=2, cbar_kws={"label": "Relative score (green = better)", "shrink": .8, "pad": .03},
            linewidths=2, linecolor="white", ax=ax, annot_kws={"size": 10})
ax.set_ylabel(""); ax.xaxis.tick_top(); ax.tick_params(length=0, labelsize=9.5)
ax.set_title("Fig 5. Mode scorecard: Bus is weakest on reliability; micromobility loses money\n", loc="left")
save("fig5_scorecard.png")

In [ ]:
# stats_tests.py: Bus reliability scenario
# 4. Scenario: Bus reliability
bus = d[d.Transport_Mode == "Bus"]; net_ot = d.On_Time_Percentage.mean()
gap = net_ot - bus.On_Time_Percentage.mean()
print(f"Bus on-time {bus.On_Time_Percentage.mean():.1f}% vs network {net_ot:.1f}% (gap {gap:.1f} pts); bus pax {bus.Passengers.sum()/1e6:.2f}M")
print(f"  extra on-time passengers (2 yrs) ~ {bus.Passengers.sum()*gap/100/1e6:.2f}M; per year ~ {bus.Passengers.sum()*gap/100/2/1e6:.2f}M")
dg = bus[D].mean() - d[D].mean(); print(f"  delay gap {dg:.2f} min -> predicted rating gain {-b*dg:.2f} points (model slope)")

In [ ]:
# stats_tests.py: micromobility losses
# 6. Micromobility losses
mm = d[d.Transport_Mode.isin(["Bike Share", "E-Scooter"])]
print(f"Micromobility net {mm.Net_Result_EUR.sum()/1e6:.2f}M; revenue {mm.Ticket_Revenue_EUR.sum()/1e6:.2f}M cost {mm.Operating_Cost_EUR.sum()/1e6:.2f}M")
print("Others net", round(d[~d.index.isin(mm.index)].Net_Result_EUR.sum() / 1e6, 2))

## A.7 Figure 6: passengers versus CO2 saved (scatter plot)

In [ ]:
# FIG 6  RELATIONSHIP (scatter, log-log)
fig, ax = plt.subplots(figsize=(9, 5.5))
mcol = dict(zip(df.Transport_Mode.cat.categories, [OI[5], OI[3], OI[4], OI[0], OI[2], OI[1]]))
sns.scatterplot(data=df, x="Passengers", y="CO2_Saved_KG", hue="Transport_Mode", palette=mcol, s=14, alpha=.55, linewidth=0, ax=ax)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Passengers per record (log scale)"); ax.set_ylabel("CO2 saved, kg (log scale)")
ax.legend(title="Mode", frameon=False, markerscale=2)
ax.set_title("Fig 6. CO2 saved scales almost linearly with ridership (r = 0.99)")
save("fig6_scatter_co2.png")

## A.8 Figure 7: delay by mode and weather (heatmap)

In [ ]:
# FIG 7  HEATMAP mode x weather delay
pv = df.pivot_table(index="Transport_Mode", columns="Weather", values="Average_Delay_Minutes", observed=True)
pv = pv.loc[mo.sort_values("Delay", ascending=False).index, order]
fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.heatmap(pv, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=1, cbar_kws={"label": "Mean delay (min)"}, ax=ax)
ax.set_xlabel(""); ax.set_ylabel("")
ax.set_title("Fig 7. Bus in snow is the single worst combination (10.3 min)")
save("fig7_delay_heatmap.png")

## A.9 Figure 8: accessibility complaints per 1,000 passengers

In [ ]:
# FIG 8  ACCESSIBILITY (normalised)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True)
net_rate = df.Accessibility_Complaints.sum() / df.Passengers.sum() * 1000
for ax, (by, ttl) in zip(axs, [("Transport_Mode", "By transport mode"), ("Route_Type", "By route type")]):
    a = agg(by).sort_values("Compl_per_1000")
    ax.barh(a.index.astype(str), a.Compl_per_1000, color=[OI[3] if v > net_rate * 1.1 else OI[5] for v in a.Compl_per_1000])
    ax.axvline(net_rate, color="k", ls="--", lw=1); ax.set_title(ttl, fontsize=11); ax.set_xlabel("Complaints per 1,000 passengers")
    for i, v in enumerate(a.Compl_per_1000): ax.text(v + .01, i, f"{v:.2f}", va="center", fontsize=9)
axs[0].text(net_rate + .01, -.75, f"network {net_rate:.2f}", fontsize=8)
fig.suptitle("Fig 8. Once volume is removed, E-Scooter, Bus and Suburban Feeder have the most accessibility complaints", fontweight="bold", fontsize=11)
save("fig8_accessibility.png")

In [ ]:
# stats_tests.py: accessibility scenario
# 5. Scenario: accessibility
net_rate = d.Accessibility_Complaints.sum() / d.Passengers.sum() * 1000
tot = 0
for mode_name in ["E-Scooter", "Bus", "Bike Share"]:   # not "mo": that name holds the scorecard table in eco_analysis.py
    x = d[d.Transport_Mode == mode_name]; r = x.Accessibility_Complaints.sum() / x.Passengers.sum() * 1000
    av = (r - net_rate) * x.Passengers.sum() / 1000; tot += av
    print(f"  {mode_name}: {r:.2f} vs {net_rate:.2f}; complaints avoided at network rate ~ {av:.0f}")
print(f"  total avoided ~ {tot:.0f} of {d.Accessibility_Complaints.sum():,} ({tot/d.Accessibility_Complaints.sum()*100:.1f}%)")

## A.10 Key figures printed for the report text

In [ ]:
# KEY NUMBERS FOR REPORT
print("\n==== KEY NUMBERS ====")
y = df.groupby("Year").agg(P=("Passengers", "sum"), D=("Average_Delay_Minutes", "mean"), OT=("On_Time_Percentage", "mean"))
print(y, "\nPassenger change 2025 vs 2024: %.1f%%" % ((y.P[2025] / y.P[2024] - 1) * 100))
print(mo[["Passengers", "OnTime", "Delay", "Rating", "Margin", "Compl_per_1000", "CO2_per_Pax", "Rev_per_Pax", "Cost_per_Pax"]].round(2))
print("Metro share of pax: %.1f%%, of CO2: %.1f%%" % (mo.Passengers["Metro"] / mo.Passengers.sum() * 100, mo.CO2["Metro"] / mo.CO2.sum() * 100))
ev = df.groupby("Event_Day").agg(P=("Passengers", "mean"), D=("Average_Delay_Minutes", "mean"), OT=("On_Time_Percentage", "mean"), R=("Customer_Rating", "mean"))
print(ev); print("Share of loss-making records: %.1f%%" % ((df.Net_Result_EUR < 0).mean() * 100))
print("Total revenue %.1fM cost %.1fM CO2 %.0f t" % (df.Ticket_Revenue_EUR.sum() / 1e6, df.Operating_Cost_EUR.sum() / 1e6, df.CO2_Saved_KG.sum() / 1000))
print(df.groupby("Weather", observed=True)[["Average_Delay_Minutes", "Customer_Rating"]].mean().round(2))
print("Rating~Delay r = %.2f" % df.Customer_Rating.corr(df.Average_Delay_Minutes))

## Optional: download all figures from Colab

In [ ]:
import shutil
shutil.make_archive("figures", "zip", "figs")
try:
    from google.colab import files
    files.download("figures.zip")
except ImportError:
    print("Figures are saved in the figs folder.")

# C. Interactive versions of Figures 1 to 8: hover over the charts

Move the mouse over a bar, point or cell: the tooltip explains what it means (for example, how a delay compares with the mode's average, or what a correlation says in plain words).

## C.1 Figure 1: passengers per record over time (hover for the story)

In [ ]:
"""COM7021 - interactive (hover-for-the-story) versions of Figures 1 to 8, using Plotly.
Run after eco_analysis.py (needs eco_clean.csv). In Colab/Jupyter each chart shows below its cell."""
import pandas as pd, numpy as np
import plotly.graph_objects as go, plotly.express as px
from plotly.subplots import make_subplots

df = pd.read_csv("eco_clean.csv", parse_dates=["Date", "Month"])
OI = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9"]          # colour-blind-safe palette
HL = dict(bgcolor="white", font=dict(size=12, color="#1F2933"), align="left", bordercolor="#9AA5B1")
base = dict(template="plotly_white", hoverlabel=HL, margin=dict(l=10, r=10, t=60, b=10))
MODE_COL = dict(zip(sorted(df.Transport_Mode.unique()), OI))
NET_OT, NET_DELAY = df.On_Time_Percentage.mean(), df.Average_Delay_Minutes.mean()


# Figure 1: passengers per record over time
m = df.groupby("Month").agg(Records=("Passengers", "size"), Pax=("Passengers", "mean"), Delay=("Average_Delay_Minutes", "mean")).reset_index()
m["Roll3"] = m.Pax.rolling(3).mean(); ov = df.Passengers.mean()
f = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[.5, .25, .25], vertical_spacing=.06)
f.add_scatter(x=m.Month, y=m.Pax, mode="lines+markers", name="Per record", line=dict(color=OI[5]), row=1, col=1,
              customdata=np.c_[(m.Pax / ov - 1) * 100, m.Records],
              hovertemplate="<b>%{x|%B %Y}</b><br>Average <b>%{y:,.0f}</b> passengers per record (%{customdata[0]:+.0f}% vs overall average)"
                            "<br>Based on %{customdata[1]} records<br><i>Totals move with the number of records, so per-record demand is the fair comparison.</i><extra></extra>")
f.add_scatter(x=m.Month, y=m.Roll3, mode="lines", name="3-month average", line=dict(color=OI[0], width=4), row=1, col=1,
              hovertemplate="<b>%{x|%B %Y}</b><br>3-month rolling average: <b>%{y:,.0f}</b> passengers per record<extra></extra>")
f.add_bar(x=m.Month, y=m.Records, name="Records", marker_color=OI[4], row=2, col=1,
          hovertemplate="<b>%{x|%B %Y}</b><br>%{y} records in the data this month<br><i>Fewer records = lower totals, even if demand is unchanged.</i><extra></extra>")
f.add_bar(x=m.Month, y=m.Delay, name="Delay", marker_color=OI[3], row=3, col=1, customdata=(m.Delay - NET_DELAY).values.reshape(-1, 1),
          hovertemplate="<b>%{x|%B %Y}</b><br>Average delay <b>%{y:.2f} min</b><br>%{customdata[0]:+.2f} min vs the 2-year average<extra></extra>")
f.update_yaxes(title_text="Pax / record", row=1, col=1); f.update_yaxes(title_text="Records", range=[150, 240], row=2, col=1)
f.update_yaxes(title_text="Delay (min)", range=[3, 5.5], row=3, col=1)
f.update_layout(**base, height=650, title="Fig 1. Passengers per record show no clear year-on-year decline", showlegend=False); f.show()

## C.2 Figure 2: passengers by city (hover for the story)

In [ ]:
# Figure 2: demand by city
c = df.groupby(["City", "Country"]).agg(Passengers=("Passengers", "sum"), per=("Passengers", "mean"), rating=("Customer_Rating", "mean")).reset_index()
c["share"] = c.Passengers / c.Passengers.sum(); c["rank"] = c.Passengers.rank(ascending=False).astype(int); c = c.sort_values("Passengers")
f = px.bar(c, x="Passengers", y="City", color="Country", orientation="h", color_discrete_sequence=OI, custom_data=["Country", "share", "rank", "per", "rating"],
           title="Fig 2. Paris and Berlin each carry about 11% of all passengers")
f.update_traces(hovertemplate="<b>%{y}</b> (%{customdata[0]})<br>%{x:,.0f} passengers = <b>%{customdata[1]:.1%}</b> of the network<br>Rank %{customdata[2]} of 12"
                              "<br>Average %{customdata[3]:,.0f} passengers per record | rating %{customdata[4]:.2f}<extra></extra>")
f.update_yaxes(categoryorder="total ascending", title=""); f.update_layout(**base, height=480); f.show()

## C.3 Figure 3: correlation heatmap (hover for the story)

In [ ]:
# Figure 3: correlation heatmap with plain-language hover
num = ["Passengers", "Trips", "Average_Delay_Minutes", "On_Time_Percentage", "Ticket_Revenue_EUR", "Operating_Cost_EUR", "CO2_Saved_KG",
       "Energy_Consumption_KWh", "Customer_Rating", "Accessibility_Complaints"]
lab = ["Passengers", "Trips", "Delay (min)", "On-time %", "Revenue", "Op. cost", "CO2 saved", "Energy", "Rating", "Access. compl."]
cm = df[num].corr().values
def story(r, same):
    if same: return "The same measure, so r = 1 by definition."
    a = abs(r); s = "Very strong" if a >= .9 else "Strong" if a >= .7 else "Moderate" if a >= .4 else "Weak" if a >= .2 else "Negligible"
    return f"{s} association: the two " + ("tend to rise together." if r > 0 else "tend to move in opposite directions.") + (" Association, not proof of cause." if a >= .2 else "")
txt = np.array([[story(cm[i, j], i == j) for j in range(10)] for i in range(10)])
f = go.Figure(go.Heatmap(z=cm, x=lab, y=lab, zmin=-1, zmax=1, colorscale="RdBu_r", customdata=txt, texttemplate="%{z:.2f}",
              hovertemplate="<b>%{y} vs %{x}</b><br>r = <b>%{z:.2f}</b><br>%{customdata}<extra></extra>", colorbar=dict(title="r")))
f.update_yaxes(autorange="reversed"); f.update_layout(**base, height=600, title="Fig 3. Volume metrics move together; delay and satisfaction move oppositely"); f.show()

## C.4 Figure 4: delay by weather (hover for the story)

In [ ]:
# Figure 4: delay by weather, with a story marker at each mean
order = df.groupby("Weather").Average_Delay_Minutes.median().sort_values().index
clear = df[df.Weather == "Clear"].Average_Delay_Minutes.mean(); f = go.Figure()
for w in order:
    f.add_box(y=df[df.Weather == w].Average_Delay_Minutes, name=w, marker_color=OI[3] if w == "Snow" else OI[0], boxpoints="outliers", showlegend=False)
mu = df.groupby("Weather").agg(mean=("Average_Delay_Minutes", "mean"), n=("Passengers", "size"), rating=("Customer_Rating", "mean")).loc[order]
f.add_scatter(x=list(order), y=mu["mean"], mode="markers", marker=dict(symbol="diamond", size=11, color="black"), showlegend=False,
              customdata=np.c_[mu["mean"] / clear, mu.n, mu.rating],
              hovertemplate="<b>%{x} days</b><br>Average delay <b>%{y:.1f} min</b> = %{customdata[0]:.1f}x the clear-weather average"
                            "<br>Average rating %{customdata[2]:.2f}/5 | %{customdata[1]:.0f} records<extra></extra>")
f.update_layout(**base, height=500, yaxis_title="Average delay (min)", title="Fig 4. Snow more than doubles typical delays (diamonds = means; hover for the story)"); f.show()

## C.5 Figure 5: mode performance scorecard (hover for the story)

In [ ]:
# Figure 5: mode scorecard, hover shows the raw value and the rank
g = df.groupby("Transport_Mode").agg(Pax=("Passengers", "sum"), OnTime=("On_Time_Percentage", "mean"), Delay=("Average_Delay_Minutes", "mean"),
        Rating=("Customer_Rating", "mean"), Rev=("Ticket_Revenue_EUR", "sum"), Cost=("Operating_Cost_EUR", "sum"), Compl=("Accessibility_Complaints", "sum"),
        CO2=("CO2_Saved_KG", "sum"))
g["Margin"] = (g.Rev - g.Cost) / g.Rev * 100; g["Compl1000"] = g.Compl / g.Pax * 1000; g["CO2pp"] = g.CO2 / g.Pax; g["PaxM"] = g.Pax / 1e6
g = g.sort_values("Pax", ascending=False)
cols = [("PaxM", "Passengers (M)", True, "{:.1f}"), ("OnTime", "On-time %", True, "{:.1f}"), ("Delay", "Avg delay (min)", False, "{:.1f}"),
        ("Rating", "Rating (1-5)", True, "{:.2f}"), ("Margin", "Margin %", True, "{:,.0f}"), ("Compl1000", "Complaints /1,000", False, "{:.2f}"),
        ("CO2pp", "CO2 saved kg/pax", True, "{:.3f}")]
Z, T, R = [], [], []
for col, name, hi, fmt in cols:
    v = g[col]; z = (v.clip(lower=-100) - v.clip(lower=-100).mean()) / v.clip(lower=-100).std(); Z.append((z if hi else -z).values)
    rk = v.rank(ascending=not hi).astype(int); T.append([fmt.format(x) for x in v]);
    R.append([("BEST of 6 modes" if r == 1 else "WORST of 6 modes" if r == 6 else f"Rank {r} of 6 (1 = best)") for r in rk])
f = go.Figure(go.Heatmap(z=np.array(Z).T, x=[c[1] for c in cols], y=list(g.index), zmin=-2, zmax=2, colorscale="BrBG", showscale=False,
              text=np.array(T).T, texttemplate="%{text}", customdata=np.array(R).T,
              hovertemplate="<b>%{y}: %{x}</b><br>Value: <b>%{text}</b><br>%{customdata}<br>Green = better than the other modes, brown = worse<extra></extra>"))
f.update_yaxes(autorange="reversed"); f.update_xaxes(side="top")
f.update_layout(**{**base, "margin": dict(l=10, r=10, t=110, b=10)}, height=470, title=dict(text="Fig 5. Bus is weakest on reliability; micromobility loses money", y=0.97)); f.show()

## C.6 Figure 6: passengers versus CO2 saved (hover for the story)

In [ ]:
# Figure 6: passengers vs CO2 saved (log-log), each point tells its story
s = df.assign(Day=df.Date.dt.strftime("%d %b %Y"))
f = px.scatter(s, x="Passengers", y="CO2_Saved_KG", color="Transport_Mode", color_discrete_map=MODE_COL, opacity=.6, log_x=True, log_y=True,
               custom_data=["Transport_Mode", "City", "Day", "CO2_per_Pax", "Weather"], render_mode="svg",
               title="Fig 6. CO2 saved scales almost linearly with ridership (r = 0.99)")
f.update_traces(marker=dict(size=5), hovertemplate="<b>%{customdata[0]}</b> in %{customdata[1]}, %{customdata[2]} (%{customdata[4]})"
                "<br>%{x:,.0f} passengers saved <b>%{y:,.0f} kg CO2</b><br>= %{customdata[3]:.3f} kg per passenger<extra></extra>")
f.update_layout(**base, height=520); f.show()

## C.7 Figure 7: delay by mode and weather (hover for the story)

In [ ]:
# Figure 7: delay by mode and weather
pv = df.pivot_table(index="Transport_Mode", columns="Weather", values="Average_Delay_Minutes", aggfunc="mean")
cnt = df.pivot_table(index="Transport_Mode", columns="Weather", values="Average_Delay_Minutes", aggfunc="size")
dev = pv.sub(df.groupby("Transport_Mode").Average_Delay_Minutes.mean(), axis=0)
pv = pv.loc[df.groupby("Transport_Mode").Average_Delay_Minutes.mean().sort_values(ascending=False).index]; cnt, dev = cnt.loc[pv.index], dev.loc[pv.index]
f = go.Figure(go.Heatmap(z=pv.values, x=list(pv.columns), y=list(pv.index), colorscale="YlOrRd", texttemplate="%{z:.1f}", customdata=np.dstack([dev.values, cnt.values]),
              colorbar=dict(title="min"), hovertemplate="<b>%{y} in %{x} weather</b><br>Average delay: <b>%{z:.1f} min</b>"
              "<br>%{customdata[0]:+.1f} min vs this mode's all-weather average<br>Based on %{customdata[1]:.0f} records<extra></extra>"))
f.update_yaxes(autorange="reversed"); f.update_layout(**base, height=450, title="Fig 7. Bus in snow is the single worst combination (10.3 min)"); f.show()

## C.8 Figure 8: accessibility complaints per 1,000 passengers (hover for the story)

In [ ]:
# ---------------- Figure 8: accessibility complaints per 1,000 passengers
net = df.Accessibility_Complaints.sum() / df.Passengers.sum() * 1000
f = make_subplots(rows=1, cols=2, subplot_titles=("By transport mode", "By route type"), shared_xaxes=True)
for k, by in enumerate(["Transport_Mode", "Route_Type"], 1):
    a = df.groupby(by).agg(c=("Accessibility_Complaints", "sum"), p=("Passengers", "sum")); a["rate"] = a.c / a.p * 1000
    a["vs"] = (a.rate / net - 1) * 100; a = a.sort_values("rate")
    f.add_bar(x=a.rate, y=a.index, orientation="h", marker_color=[OI[3] if r > net * 1.1 else OI[5] for r in a.rate], showlegend=False,
              customdata=np.c_[a.vs, a.c, a.p], row=1, col=k,
              hovertemplate="<b>%{y}</b><br><b>%{x:.2f}</b> complaints per 1,000 passengers<br>%{customdata[0]:+.0f}% vs the network rate (" + f"{net:.2f}" + ")"
                            "<br>%{customdata[1]:,.0f} complaints from %{customdata[2]:,.0f} passengers<br><i>A rate, so busy services are not penalised for volume.</i><extra></extra>")
    f.add_vline(x=net, line_dash="dash", row=1, col=k)
f.update_xaxes(title_text="Complaints per 1,000 passengers")
f.update_layout(**base, height=430, title="Fig 8. Relative to demand, E-Scooter, Bus and Suburban Feeder draw the most accessibility complaints"); f.show()

# B. Interactive dashboard
Use the filters, click a bar in the mode or city chart to drill down, and hover over any chart or KPI card for a plain-language explanation.

In [ ]:
#EcoMove Mobility interactive dashboard (Plotly Dash).
import sys, pandas as pd, numpy as np
import plotly.express as px, plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, no_update
from plotly.subplots import make_subplots

# DATA is set in the setup cell above
df = pd.read_excel(DATA)
df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30")
df["Year"] = df.Date.dt.year
df["Month"] = df.Date.dt.to_period("M").dt.to_timestamp()

OI = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9"]   # colour-blind-safe (Okabe-Ito)
MODE_COL = dict(zip(sorted(df.Transport_Mode.unique()), OI))
BG, CARD, INK = "#F4F6F8", "#FFFFFF", "#1F2933"
LAYOUT = dict(template="plotly_white", margin=dict(l=10, r=10, t=40, b=10), font=dict(size=12, color=INK),
              title=dict(font=dict(size=14), x=0.01),
              hoverlabel=dict(bgcolor="white", font=dict(size=12, color=INK), align="left", bordercolor="#9AA5B1"))

def opts(col): return [{"label": v, "value": v} for v in sorted(df[col].unique())]

def dd(id_, label, col, multi=True):
    return html.Div([html.Label(label, style={"fontSize": 12, "fontWeight": 600}),
                     dcc.Dropdown(id=id_, options=opts(col), multi=multi, placeholder="All")],
                    style={"flex": "1", "minWidth": "150px"})

TIPS = {"k_pax": "Total recorded passengers in the current selection. Totals depend on how many route-day records are included.",
        "k_ot": "Average share of services running on time. The line underneath compares it with the whole network.",
        "k_delay": "Average delay per record in minutes. Lower is better; snow and event days push it up.",
        "k_rate": "Average customer rating out of 5. Across the network it moves closely with delay (r = -0.82), an association, not proof of cause.",
        "k_net": "Ticket revenue minus operating cost. Negative means the selection loses money overall.",
        "k_co2": "CO2 saved by passengers using public or shared transport instead of other options. Mostly Metro.",
        "k_acc": "Accessibility complaints per 1,000 passengers. Using a rate stops busy services looking worse just because they are busy."}

def kpi(id_, label):
    return html.Div(title=TIPS.get(id_, ""), children=[html.Div(label, style={"fontSize": 12, "color": "#52606D"}),
                     html.Div(id=id_, style={"fontSize": 24, "fontWeight": 700}),
                     html.Div(id=id_ + "_sub", style={"fontSize": 11, "color": "#52606D"})],
                    style={"background": CARD, "padding": "10px 14px", "borderRadius": 8, "flex": "1",
                           "boxShadow": "0 1px 3px rgba(0,0,0,.12)", "minWidth": "130px", "cursor": "help"})

card = lambda g: html.Div(g, style={"background": CARD, "borderRadius": 8, "padding": 6, "boxShadow": "0 1px 3px rgba(0,0,0,.12)"})

app = Dash(__name__)
app.title = "EcoMove Mobility Dashboard"
app.layout = html.Div(style={"background": BG, "padding": 16, "fontFamily": "Segoe UI, Arial, sans-serif", "color": INK}, children=[
    html.H2("EcoMove City Transport Authority - Sustainable Mobility Dashboard", style={"margin": "0 0 2px"}),
    html.Div("Jan 2024 - Dec 2025 | 12 cities | 6 countries | 6 transport modes. Use the filters, or click a bar in the mode or city chart, to focus every chart and KPI.",
             style={"fontSize": 13, "color": "#52606D", "marginBottom": 10}),
    html.Div([dd("f_country", "Country", "Country"), dd("f_city", "City", "City"), dd("f_mode", "Transport mode", "Transport_Mode"),
              dd("f_route", "Route type", "Route_Type"), dd("f_weather", "Weather", "Weather"),
              html.Div([html.Label("Event day", style={"fontSize": 12, "fontWeight": 600}),
                        dcc.RadioItems(id="f_event", options=[{"label": " All", "value": "All"}, {"label": " Event", "value": "Yes"},
                                                              {"label": " Non-event", "value": "No"}], value="All", inline=True,
                                       inputStyle={"marginLeft": 8})], style={"minWidth": "190px"}),
              html.Div([html.Label("Year", style={"fontSize": 12, "fontWeight": 600}),
                        dcc.Checklist(id="f_year", options=[2024, 2025], value=[2024, 2025], inline=True,
                                      inputStyle={"marginLeft": 8})], style={"minWidth": "130px"})],
             style={"display": "flex", "gap": 12, "flexWrap": "wrap", "background": CARD, "padding": 10, "borderRadius": 8,
                    "boxShadow": "0 1px 3px rgba(0,0,0,.12)", "marginBottom": 10}),
    html.Div([kpi("k_pax", "Passengers"), kpi("k_ot", "On-time performance"), kpi("k_delay", "Avg delay (min)"),
              kpi("k_rate", "Customer rating"), kpi("k_net", "Net result (revenue - cost)"),
              kpi("k_co2", "CO2 saved (tonnes)"), kpi("k_acc", "Access. complaints /1,000 pax")],
             style={"display": "flex", "gap": 10, "flexWrap": "wrap", "marginBottom": 10}),
    html.Div(id="insight", style={"background": "#FFF8E1", "borderLeft": "4px solid #E69F00", "padding": "8px 12px",
                                  "borderRadius": 4, "fontSize": 13, "marginBottom": 10}),
    html.Div([card(dcc.Graph(id="g_trend", style={"height": 300})), card(dcc.Graph(id="g_mode", style={"height": 300}))],
             style={"display": "grid", "gridTemplateColumns": "3fr 2fr", "gap": 10, "marginBottom": 10}),
    html.Div([card(dcc.Graph(id="g_city", style={"height": 330})), card(dcc.Graph(id="g_heat", style={"height": 330})),
              card(dcc.Graph(id="g_scatter", style={"height": 330}))],
             style={"display": "grid", "gridTemplateColumns": "1fr 1fr 1fr", "gap": 10, "marginBottom": 10}),
    html.Div([card(dcc.Graph(id="g_acc", style={"height": 300})), card(dcc.Graph(id="g_box", style={"height": 300}))],
             style={"display": "grid", "gridTemplateColumns": "1fr 1fr", "gap": 10}),
])

@app.callback(Output("f_mode", "value"), Input("g_mode", "clickData"), prevent_initial_call=True)
def click_mode(cd):
    return [cd["points"][0]["y"]] if cd else no_update

@app.callback(Output("f_city", "value"), Input("g_city", "clickData"), prevent_initial_call=True)
def click_city(cd):
    return [cd["points"][0]["y"]] if cd else no_update

def filt(country, city, mode, route, weather, event, year):
    d = df
    for col, v in [("Country", country), ("City", city), ("Transport_Mode", mode), ("Route_Type", route), ("Weather", weather)]:
        if v: d = d[d[col].isin(v)]
    if event in ("Yes", "No"): d = d[d.Event_Day == event]
    return d[d.Year.isin(year or [2024, 2025])]

def empty(msg="No data for this selection"):
    f = go.Figure(); f.update_layout(**LAYOUT, annotations=[dict(text=msg, showarrow=False, font=dict(size=14))])
    f.update_xaxes(visible=False); f.update_yaxes(visible=False); return f

@app.callback(
    [Output(i, "children") for i in ["k_pax", "k_pax_sub", "k_ot", "k_ot_sub", "k_delay", "k_delay_sub", "k_rate", "k_rate_sub",
                                     "k_net", "k_net_sub", "k_co2", "k_co2_sub", "k_acc", "k_acc_sub", "insight"]] +
    [Output(g, "figure") for g in ["g_trend", "g_mode", "g_city", "g_heat", "g_scatter", "g_acc", "g_box"]],
    [Input(i, "value") for i in ["f_country", "f_city", "f_mode", "f_route", "f_weather", "f_event", "f_year"]])
def update(country, city, mode, route, weather, event, year):
    d = filt(country, city, mode, route, weather, event, year)
    if d.empty:
        return ["-"] * 14 + ["No records match the selected filters."] + [empty()] * 7
    pax = d.Passengers.sum(); rev = d.Ticket_Revenue_EUR.sum(); cost = d.Operating_Cost_EUR.sum()
    base = df[df.Year.isin(year or [2024, 2025])]
    b_ot, b_dl, b_rt = base.On_Time_Percentage.mean(), base.Average_Delay_Minutes.mean(), base.Customer_Rating.mean()
    b_acc = base.Accessibility_Complaints.sum() / base.Passengers.sum() * 1000
    acc = d.Accessibility_Complaints.sum() / pax * 1000
    sub = lambda v, b, u="": f"{v - b:+.2f}{u} vs network"
    k = [f"{pax/1e6:.2f}M", f"{len(d):,} records", f"{d.On_Time_Percentage.mean():.1f}%", sub(d.On_Time_Percentage.mean(), b_ot, " pts"),
         f"{d.Average_Delay_Minutes.mean():.2f}", sub(d.Average_Delay_Minutes.mean(), b_dl), f"{d.Customer_Rating.mean():.2f} / 5",
         sub(d.Customer_Rating.mean(), b_rt), f"EUR {(rev-cost)/1e6:+.2f}M", f"margin {(rev-cost)/rev*100:+.1f}%",
         f"{d.CO2_Saved_KG.sum()/1000:,.0f} t", f"{d.CO2_Saved_KG.sum()/pax*1000:.0f} g per passenger",
         f"{acc:.2f}", sub(acc, b_acc)]

    # ---- insight text: worst mode on delay inside current selection
    gm = d.groupby("Transport_Mode").agg(delay=("Average_Delay_Minutes", "mean"), p=("Passengers", "sum"))
    gc = d.groupby("City").Passengers.sum()
    txt = (f"In this selection {gc.idxmax()} is the highest-demand city ({gc.max()/pax:.0%} of passengers) and "
           f"{gm.delay.idxmax()} has the longest average delay ({gm.delay.max():.1f} min).")
    if d.Event_Day.nunique() > 1:
        e = d.groupby("Event_Day").Average_Delay_Minutes.mean(); txt += f" Event days average {e['Yes']-e['No']:+.1f} min more delay than non-event days."

    # ---- 1 trend (hover tells how each month compares)
    t = d.groupby("Month").agg(Passengers=("Passengers", "mean"), OT=("On_Time_Percentage", "mean"), n=("Passengers", "size")).reset_index()  # mean per record: totals depend on record counts
    t["dev"] = (t.Passengers / t.Passengers.mean() - 1) * 100
    f1 = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.62, 0.38], vertical_spacing=0.08)
    f1.add_bar(x=t.Month, y=t.Passengers, name="Passengers per record", marker_color="#56B4E9", row=1, col=1, customdata=t[["dev", "n"]].values,
               hovertemplate="<b>%{x|%B %Y}</b><br>Average passengers per record: <b>%{y:,.0f}</b><br>%{customdata[0]:+.0f}% vs this selection's monthly average"
                             "<br>Based on %{customdata[1]} records<br><i>Per-record values stay fair even when record counts change.</i><extra></extra>")
    f1.add_scatter(x=t.Month, y=t.OT, name="On-time %", line=dict(color="#D55E00", width=3), row=2, col=1, customdata=(t.OT - b_ot).values.reshape(-1, 1),
                   hovertemplate="<b>%{x|%B %Y}</b><br>On time: <b>%{y:.1f}%</b><br>%{customdata[0]:+.1f} pts vs network average<extra></extra>")
    f1.update_layout(**LAYOUT, title_text="Passengers per record (top) and on-time % (bottom)", showlegend=False, hovermode="closest")
    f1.update_yaxes(title_text="Pax / record", row=1, col=1); f1.update_yaxes(title_text="On-time %", row=2, col=1, range=[60, 100])
    # ---- 2 mode performance (hover says whether the mode earns or loses money per passenger)
    m = d.groupby("Transport_Mode").agg(Pax=("Passengers", "sum"), Rev=("Ticket_Revenue_EUR", "sum"), Cost=("Operating_Cost_EUR", "sum"),
                                         OT=("On_Time_Percentage", "mean"), Rate=("Customer_Rating", "mean")).reset_index()
    m["Net"] = (m.Rev - m.Cost) / 1e6; m = m.sort_values("Net")
    m["story"] = np.where(m.Net < 0, "Loses money: costs EUR " + (m.Cost / m.Pax).round(2).astype(str) + " per passenger against EUR " + (m.Rev / m.Pax).round(2).astype(str) + " revenue",
                          "Earns EUR " + ((m.Rev - m.Cost) / m.Pax).round(2).astype(str) + " per passenger after costs, helping fund weaker modes")
    f2 = px.bar(m, x="Net", y="Transport_Mode", orientation="h", color="Transport_Mode", color_discrete_map=MODE_COL,
                custom_data=["OT", "Rate", "story", "Rev", "Cost"], title="Net result by mode (EUR M)")
    f2.update_traces(hovertemplate="<b>%{y}</b><br>Net result: <b>EUR %{x:.2f}M</b> (revenue %{customdata[3]:,.0f}, cost %{customdata[4]:,.0f})"
                                   "<br>%{customdata[2]}<br>On time %{customdata[0]:.1f}% | rating %{customdata[1]:.2f}/5<br><i>Click to filter the dashboard to this mode</i><extra></extra>")
    f2.update_layout(**LAYOUT, showlegend=False, xaxis_title="", yaxis_title="")
    # ---- 3 city demand (hover gives share, rank and demand per record)
    c = d.groupby(["City", "Country"]).agg(Passengers=("Passengers", "sum"), per=("Passengers", "mean"), rating=("Customer_Rating", "mean")).reset_index()
    c["share"] = c.Passengers / c.Passengers.sum(); c["rank"] = c.Passengers.rank(ascending=False).astype(int); c["of"] = len(c)
    c = c.sort_values("Passengers")
    f3 = px.bar(c, x="Passengers", y="City", color="Country", orientation="h", color_discrete_sequence=OI, title="Demand by city",
                custom_data=["Country", "share", "rank", "of", "per", "rating"])
    f3.update_traces(hovertemplate="<b>%{y}</b> (%{customdata[0]})<br>%{x:,.0f} passengers = <b>%{customdata[1]:.1%}</b> of this selection"
                                   "<br>Rank %{customdata[2]} of %{customdata[3]} cities<br>Average %{customdata[4]:,.0f} passengers per record | rating %{customdata[5]:.2f}"
                                   "<br><i>Click to filter the dashboard to this city</i><extra></extra>")
    f3.update_yaxes(categoryorder="total ascending")
    f3.update_layout(**LAYOUT, legend=dict(orientation="h", y=-0.15, font=dict(size=10)), yaxis_title="", xaxis_title="")
    # ---- 4 delay heatmap (hover compares each weather with the mode's own average)
    pv = d.pivot_table(index="Transport_Mode", columns="Weather", values="Average_Delay_Minutes", aggfunc="mean")
    cnt = d.pivot_table(index="Transport_Mode", columns="Weather", values="Average_Delay_Minutes", aggfunc="size")
    dev = pv.sub(d.groupby("Transport_Mode").Average_Delay_Minutes.mean(), axis=0)
    f4 = go.Figure(go.Heatmap(z=pv.values, x=list(pv.columns), y=list(pv.index), colorscale="YlOrRd", showscale=False, texttemplate="%{z:.1f}",
                              customdata=np.dstack([dev.values, cnt.values]),
                              hovertemplate="<b>%{y} in %{x} weather</b><br>Average delay: <b>%{z:.1f} min</b>"
                                            "<br>%{customdata[0]:+.1f} min vs this mode's all-weather average<br>Based on %{customdata[1]:.0f} records<extra></extra>"))
    f4.update_layout(**LAYOUT, title_text="Avg delay (min): mode x weather", xaxis_title="", yaxis_title="")
    # ---- 5 scatter delay vs rating (hover reads each point as a short story)
    s = d.sample(min(len(d), 1500), random_state=1).assign(Day=lambda x: x.Date.dt.strftime("%d %b %Y"))
    f5 = px.scatter(s, x="Average_Delay_Minutes", y="Customer_Rating", color="Transport_Mode", color_discrete_map=MODE_COL, opacity=.55,
                    custom_data=["Transport_Mode", "City", "Weather", "Event_Day", "Day"],
                    trendline="ols", trendline_scope="overall", trendline_color_override="black", title="Delay vs customer rating", render_mode="svg")
    f5.update_traces(selector=dict(mode="markers"), hovertemplate="<b>%{customdata[0]}</b> in %{customdata[1]}, %{customdata[4]}"
                     "<br>Weather: %{customdata[2]} | event day: %{customdata[3]}<br>Delay <b>%{x:.1f} min</b> -> rating <b>%{y:.1f}/5</b>"
                     "<br><i>Slower journeys tend to be rated lower across the network (association, not proof of cause)</i><extra></extra>")
    f5.update_layout(**LAYOUT, legend=dict(orientation="h", y=-0.2, font=dict(size=10), title_text=""),
                     xaxis_title="Avg delay (min)", yaxis_title="Rating")
    # ---- 6 accessibility per 1000 (hover compares with the selection average)
    a = d.groupby("Route_Type").agg(c=("Accessibility_Complaints", "sum"), p=("Passengers", "sum"))
    a["rate"] = a.c / a.p * 1000; a["vs"] = (a.rate / acc - 1) * 100; a = a.sort_values("rate").reset_index()
    f6 = px.bar(a, x="rate", y="Route_Type", orientation="h", title="Accessibility complaints per 1,000 passengers",
                color_discrete_sequence=["#0072B2"], custom_data=["c", "p", "vs"])
    f6.update_traces(hovertemplate="<b>%{y}</b><br><b>%{x:.2f}</b> complaints per 1,000 passengers<br>%{customdata[2]:+.0f}% vs selection average"
                                   "<br>%{customdata[0]:,.0f} complaints from %{customdata[1]:,.0f} passengers<br><i>A rate, so busy routes are not penalised for volume</i><extra></extra>")
    f6.add_vline(x=acc, line_dash="dash", annotation_text="selection avg", annotation_position="top")
    f6.update_layout(**LAYOUT, xaxis_title="", yaxis_title="")
    # ---- 7 box delay by event/weather (hover on the box shows median and quartiles)
    f7 = px.box(d, x="Weather", y="Average_Delay_Minutes", color="Event_Day",
                color_discrete_map={"No": "#56B4E9", "Yes": "#D55E00"}, title="Delay distribution: weather and event days")
    f7.update_layout(**LAYOUT, yaxis_title="Delay (min)", xaxis_title="", legend=dict(orientation="h", y=1.15, x=0.5, title_text="Event day"))
    return k + [txt, f1, f2, f3, f4, f5, f6, f7]

if __name__ == "__main__":
    if "ipykernel" in sys.modules:          # Google Colab
        import socket
        def _free_port(start=8050):
            for p in range(start, start + 30):
                with socket.socket() as s_:
                    if s_.connect_ex(("127.0.0.1", p)) != 0: return p

        port = _free_port()

        try:
            from google.colab import output
            output.serve_kernel_port_as_window(port)  # External clickable link
        except ImportError:
            pass

        app.run(jupyter_mode="inline", height=1500, port=port, debug=False)  # Inline view
    else:                                   # Local script
        app.run(debug=False, port=8050)

